# نسخة Colab المبسطة جدًا — صورة إلى مجسم 3D بـ TripoSR

## هذه النسخة مخصصة للجوال وبأقل تعقيد ممكن.

### ماذا ستفعل؟
1. تفتح هذا الملف في **Google Colab**
2. تغيّر نوع التشغيل إلى **T4 GPU**
3. تشغّل خلية فحص الـGPU
4. تشغّل خلية التثبيت
5. تشغّل خلية رفع الصورة
6. تشغّل خلية التوليد
7. تشغّل خلية التحميل

---

## مهم
- أول مرة ستكون **بطيئة** لأن Colab سيحمّل النموذج.
- بعد ذلك سيصنع لك ملف **GLB**.
- إذا كانت الصورة واضحة والخلفية بسيطة، النتيجة تكون أفضل.

In [ ]:
# 1) تأكد من تفعيل GPU
import torch, platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU غير مفعّل. من Colab اختر: Runtime > Change runtime type > T4 GPU ثم أعد تشغيل الخلية."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("✅ ممتاز، الـ GPU جاهز")

In [ ]:
# 2) تثبيت TripoSR (شغّل هذه الخلية مرة واحدة فقط)
import os, subprocess, sys, pathlib

os.chdir("/content")

if not pathlib.Path("/content/TripoSR").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/VAST-AI-Research/TripoSR.git"],
        check=True
    )

os.chdir("/content/TripoSR")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "setuptools", "wheel", "ninja"],
    check=True
)

packages = [
    "omegaconf==2.3.0",
    "Pillow==10.1.0",
    "einops==0.7.0",
    "transformers==4.35.0",
    "trimesh==4.0.5",
    "rembg",
    "huggingface-hub",
    "imageio[ffmpeg]",
    "xatlas==0.0.9",
    "moderngl==5.10.0",
    "onnxruntime",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/tatsy/torchmcubes.git"],
    check=True
)

print("✅ تم تثبيت كل شيء بنجاح")

In [ ]:
# 3) ارفع الصورة من هاتفك
from google.colab import files
from pathlib import Path
import os

os.chdir("/content/TripoSR")

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("لم يتم رفع أي صورة")

input_name = next(iter(uploaded.keys()))
input_path = Path("/content/TripoSR") / input_name

print("✅ تم رفع الصورة:")
print(input_path)

In [ ]:
# 4) توليد المجسم
import os, subprocess, sys, pathlib, shutil

# غيّر هذه القيم إذا أردت
MC_RESOLUTION = 256      # 192 أسرع - 256 متوازن - 320 أدق لكن أثقل
FOREGROUND_RATIO = 0.85
CHUNK_SIZE = 8192

os.chdir("/content/TripoSR")

output_dir = pathlib.Path("/content/my_3d_output")
if output_dir.exists():
    shutil.rmtree(output_dir)

cmd = [
    sys.executable, "run.py",
    str(input_path),
    "--output-dir", str(output_dir),
    "--model-save-format", "glb",
    "--mc-resolution", str(MC_RESOLUTION),
    "--chunk-size", str(CHUNK_SIZE),
    "--foreground-ratio", str(FOREGROUND_RATIO),
]

print("⏳ جاري بناء المجسم... قد يستغرق عدة دقائق")
subprocess.run(cmd, check=True)

result_glb = output_dir / "0" / "mesh.glb"
if not result_glb.exists():
    raise RuntimeError("انتهى التشغيل لكن لم أجد ملف mesh.glb")

print("✅ تم إنشاء المجسم بنجاح:")
print(result_glb)
print("الحجم:", round(result_glb.stat().st_size / 1024 / 1024, 2), "MB")

In [ ]:
# 5) تحميل المجسم إلى هاتفك
from google.colab import files
files.download(str(result_glb))
print("⬇️ بدأ تنزيل ملف GLB")

# إذا حصل خطأ أو كان بطيئًا

## جرّب هذه التعديلات:
- اجعل `MC_RESOLUTION = 192`
- استخدم صورة أصغر أو أوضح
- تأكد من بقاء Colab مفتوحًا أثناء المعالجة

## إذا كنت تريد جودة أعلى:
- ارفع `MC_RESOLUTION` إلى 320
- لكن هذا أبطأ ويستهلك GPU أكثر